# Pipeline di Classificazione Curriculum Vitae (Resume)

Questo notebook implementa una pipeline di Machine Learning per classificare i CV in categorie.

**Step:**
1. Caricamento e Pulizia Dati
2. Split del Dataset (Train / Validation / Test)
3. Feature Extraction (TF-IDF)
4. Addestramento Modello (Linear SVM)
5. Valutazione
6. Inferenza

In [2]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import re
import seaborn as sns
import string

## 1. Caricamento e Preprocessing dei Dati

In [ ]:
df = pd.read_csv('data/Resume.csv')

print(f"Dataset caricato con successo: {df.shape[0]} righe, {df.shape[1]} colonne")
def clean_text(text):
    s = str(text)
    tokens = s.split()
    i = 0

    # Rimozione delle parole iniziali tutte maiuscole
    while i < len(tokens):
        letters = re.sub(r'[^A-Za-z]', '', tokens[i])
        if letters and letters.isupper():
            i += 1
            continue
        break
    trimmed = ' '.join(tokens[i:]) if i < len(tokens) else s

    trimmed = trimmed.lower()
    trimmed = trimmed.translate(str.maketrans('', '', string.punctuation))
    trimmed = re.sub(r'\s+', ' ', trimmed).strip()  # Rimuove spazi extra
    return trimmed

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

# Visualizzazione delle classi
print("\nDistribuzione delle Categorie:")
print(df['Category'].value_counts().head())

Dataset caricato con successo: 2484 righe, 4 colonne


TypeError: expected string or bytes-like object

In [ ]:



def top_words_from_series(series, n=30):
    corpus = ' '.join(series.dropna().astype(str))
    tokens = re.findall(r'\b[a-z]+\b', corpus)
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return Counter(tokens).most_common(n)

# Overall top words
common = top_words_from_series(df['cleaned_resume'], 30)
if common:
    top = pd.DataFrame(common, columns=['word','count'])
    fig = px.bar(top, x='word', y='count', title='Top 30 parole (cleaned, stopwords rimosse)', labels={'count':'Count','word':'Word'})
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_image("./data/wordcloud/wordcloud_overall.png")

else:
    print('Nessun token trovato nel corpus.')

# Top words per categoria (mostra separatamente)
for cat in df['Category'].unique():
    common = top_words_from_series(df.loc[df['Category'] == cat, 'cleaned_resume'], 20)
    if not common:
        continue
    top = pd.DataFrame(common, columns=['word','count'])
    fig = px.bar(top, x='word', y='count', title=f'Top parole - {cat}', labels={'count':'Count','word':'Word'})
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_image(f"./data/wordcloud/wordcloud_{cat}.png")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [35]:

# Grafico interattivo della distribuzione delle categorie
counts = df['Category'].value_counts().reset_index()
counts.columns = ['Category','Count']
fig = px.bar(counts, x='Category', y='Count', color='Category',
             title='Distribuzione delle Categorie', labels={'Count':'Conteggio'})
fig.show()

## 2. Suddivisione del Dataset (Train + Validation + Test)

Suddividiamo i dati in:
- **Train Set (70%)**: Per addestrare il modello.
- **Validation Set (15%)**: Per il tuning degli iperparametri (implicito in questa pipeline semplificata).
- **Test Set (15%)**: Per la valutazione finale imparziale.

In [42]:
X = df['cleaned_resume']
y = df['Category']

# Primo split: Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Secondo split: Temp in Validation (15% orig) e Test (15% orig)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Dimensioni Train: {X_train.shape[0]}")
print(f"Dimensioni Validation: {X_val.shape[0]}")
print(f"Dimensioni Test: {X_test.shape[0]}")

Dimensioni Train: 1738
Dimensioni Validation: 373
Dimensioni Test: 373


## 3. Feature Extraction
Utilizziamo **TF-IDF (Term Frequency - Inverse Document Frequency)**. 
È molto efficace per il testo perché penalizza le parole troppo comuni (stop words) e valorizza quelle distintive per ogni categoria.

In [43]:
# Inizializzazione del vettorizzatore
# max_features=5000 limita il vocabolario alle 5000 parole più importanti per ridurre la dimensionalità
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))

# Fit solo sul training set per evitare data leakage
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(f"Shape della matrice di feature (Train): {X_train_vec.shape}")

Shape della matrice di feature (Train): (1738, 5000)


## 4. Addestramento del Classificatore
Utilizziamo una **Linear SVC (Support Vector Classifier)**, nota per essere veloce e accurata nella classificazione di testi.

In [44]:
model = LinearSVC(random_state=42, dual='auto')
model.fit(X_train_vec, y_train)
print("Addestramento completato.")

Addestramento completato.


## 5. Valutazione del Modello
Valutiamo le performance sul **Test Set**.

In [45]:
y_pred = model.predict(X_test_vec)

print("--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred))

print(f"Accuracy finale: {accuracy_score(y_test, y_pred):.4f}")

--- Classification Report (Test Set) ---
                        precision    recall  f1-score   support

            ACCOUNTANT       0.53      0.94      0.68        17
              ADVOCATE       0.55      0.71      0.62        17
           AGRICULTURE       0.80      0.40      0.53        10
               APPAREL       0.64      0.50      0.56        14
                  ARTS       0.50      0.38      0.43        16
            AUTOMOBILE       1.00      0.20      0.33         5
              AVIATION       0.88      0.78      0.82        18
               BANKING       0.76      0.72      0.74        18
                   BPO       1.00      0.25      0.40         4
  BUSINESS-DEVELOPMENT       0.67      0.89      0.76        18
                  CHEF       0.79      0.88      0.83        17
          CONSTRUCTION       0.93      0.82      0.88        17
            CONSULTANT       0.40      0.22      0.29        18
              DESIGNER       0.69      0.69      0.69        1

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig = px.imshow(cm, x=labels, y=labels, color_continuous_scale='Blues',
                labels=dict(x='Predetto', y='Reale', color='Count'),
                text_auto=True, title='Matrice di Confusione', height=800, width=800)
fig.update_layout(xaxis_title='Predetto', yaxis_title='Reale')
fig.show()

## 6. Inferenza
Utilizziamo il modello addestrato su un nuovo testo mai visto.

In [87]:
# Seleziona una riga casuale e ottieni testo e label reale
sample_row = df.sample(1).iloc[0]
resume_text = sample_row['cleaned_resume']
true_label = sample_row['Category']

# Preprocessing e vettorizzazione del campione
sample_clean = clean_text(resume_text)
sample_vec = vectorizer.transform([sample_clean])

# Predizione
prediction = model.predict(sample_vec)[0]

print(f"Testo Input:\n{resume_text.strip()}")
print(f"\nCategoria Predetta: {prediction}")
print(f"Categoria Reale: {true_label}")

Testo Input:
summary 15 years of leadership experience in information technology as an it director and consultant extensive strategic vendor management expertise vmo leadership expert in vendor selection process rfi rfp msa and sow and leader in contract negotiations senior project management leadership cochairman of change management review board saved millions of dollars in vendor expenses through successfully implemented sourcing partnerships implemented and lead a business relationship management team accomplished it technologist with a strong business acumen including an mba degree successfully resolved complex business technical and operational issues specialist at presenting executive level technical business presentations vpsvpcio highlights global and strategic sourcing negotiations expert vendor management project management vendor selection process it technical support cloud computing mba degree experience information technology senior manager april 2013 to february 2015 com